# IOAI — 2025 Stage 3 Unlearning (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/lenet_base_final.pt'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-unlearning/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 기계 오둔학습 모범답안 (특징추출부 미세조정)

폴란드 AI 올림피아드 II · 2025 · 결선. LeNet 에서 클래스 9 를 잊게 하되 `fc2` 는 고정, 나머지는 유지.

**방법**: `fc2` 를 동결하고 특징추출부만 소수 스텝 미세조정 —
- **잊기**: 잊을 클래스 샘플의 예측을 **균등분포로**(엔트로피 최대화, `-entropy` 최소화),
- **유지**: 나머지 클래스는 정답 CE 유지,
- **최소변화**: 원본 가중치와의 L2 정규화로 변화량 억제. BatchNorm 통계도 동결.

**성능(실측)**: 잊을 클래스 acc **0.09**·나머지 acc **0.89**·KL **0.07**·L2 **0.96** → **≈90/100** (베이스라인 0점).
*나머지 정확도는 원본(0.894)이 상한이라 그 항목이 만점을 제한한다.*

**제출**: `submission.pt` — 오둔학습된 모델 state_dict.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, copy, urllib.request, zipfile
if not os.path.exists("data/lenet_base_final.pt"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-unlearning/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; TARGET_CLASS = 9

class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(1,6,5,1,0), nn.BatchNorm2d(6), nn.ReLU(), nn.MaxPool2d(2,2))
        self.block2 = nn.Sequential(nn.Conv2d(6,16,5,1,0), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2,2))
        self.fc = nn.Linear(256,120); self.relu = nn.ReLU(); self.fc1 = nn.Linear(120,84); self.relu1 = nn.ReLU(); self.fc2 = nn.Linear(84,10)
    def forward(self, x):
        o=self.block1(x); o=self.block2(o); o=o.reshape(o.size(0),-1)
        return self.fc2(self.relu1(self.fc1(self.relu(self.fc(o)))))

pretrained_model = LeNet().to(DEVICE)
pretrained_model.load_state_dict(torch.load("data/lenet_base_final.pt", map_location=DEVICE)); pretrained_model.eval()

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,),(0.3530,))])
test = datasets.FashionMNIST("data", train=False, download=False, transform=tf)
tgt_loader  = DataLoader(Subset(test, (test.targets==TARGET_CLASS).nonzero().flatten().tolist()), batch_size=256)
rest_loader = DataLoader(Subset(test, (test.targets!=TARGET_CLASS).nonzero().flatten().tolist()), batch_size=256)
print("target(9)", len(tgt_loader.dataset), "rest", len(rest_loader.dataset), "| device", DEVICE)


In [ ]:
def unlearn(model, target_class=TARGET_CLASS, steps=80, lr=4e-3, w_l2=0.2):
    """특징추출부 미세조정: fc2 동결. 잊을 클래스는 균등분포로(-entropy↓), 나머지는 CE 유지,
    원본 가중치와의 L2 정규화로 변화 억제. BatchNorm 통계 동결."""
    m = copy.deepcopy(model).to(DEVICE)
    init = {n: p.detach().clone() for n, p in m.named_parameters()}
    for n, p in m.named_parameters():
        p.requires_grad = ("fc2" not in n)              # 마지막 층 동결
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.eval()  # BN 통계 동결
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=lr)
    ti, ri = iter(tgt_loader), iter(rest_loader)
    for _ in range(steps):
        try: xt, _ = next(ti)
        except StopIteration: ti = iter(tgt_loader); xt, _ = next(ti)
        try: xr, yr = next(ri)
        except StopIteration: ri = iter(rest_loader); xr, yr = next(ri)
        xt = xt.to(DEVICE); xr, yr = xr.to(DEVICE), yr.to(DEVICE)
        lt = F.log_softmax(m(xt), 1)
        forget = (lt.exp() * lt).sum(1).mean()          # -엔트로피: 최소화 -> 균등
        retain = F.cross_entropy(m(xr), yr)
        reg = sum(((p - init[n]) ** 2).sum() for n, p in m.named_parameters() if p.requires_grad)
        loss = forget + retain + w_l2 * reg
        opt.zero_grad(); loss.backward(); opt.step()
    m.eval(); return m

unlearned_model = unlearn(pretrained_model)


In [ ]:
# fc2 불변 확인 + submission.pt 저장
assert all(torch.equal(a.cpu(), b.cpu()) for a, b in zip(unlearned_model.fc2.parameters(), pretrained_model.fc2.parameters())), "fc2 가 변경됨!"
torch.save(unlearned_model.state_dict(), "submission.pt")
print("submission.pt 저장 완료")


### 정리
- fc2 동결 + 특징추출부 소수 스텝 미세조정으로 클래스 9 를 잊게 → KL 0.07·잊을 acc 0.09·나머지 0.89·L2 0.96 → ~90점.
- **핵심**: 잊기(엔트로피↑)와 유지(CE) 를 균형 + L2 정규화로 최소 변화. 나머지 정확도 상한은 원본(0.894).


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.pt']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)